# GRU para formación de colas (CAETEC)

Escalera de experimentos para llegar a la GRU final y su evaluación en test, que predice `pct_posible_cola` del siguiente bloque de 30 min de cada robot.

**Reglas (iguales para todos los experimentos):**

- **Split cronológico 70 / 15 / 15**, como en *Diseño de Pruebas* y la LSTM: se ordena el dataset por `bloque_tiempo` y `ms` y se corta por fila. Cada ventana se asigna al conjunto de su bloque objetivo, así el test tiene exactamente los mismos 26,978 bloques que los demás modelos.
- **Se entrena solo con train (70 %).** El imputer y el scaler se ajustan solo con train.
- **Todo se decide con validación (15 %):** early stopping, umbral de decisión y qué experimento gana. **El test no se toca** hasta el modelo final.
- Cada paso parte del ganador del anterior y cambia **una sola cosa** (Greff et al., 2017). Gana el F1 de validación más alto; si otro queda a menos de `TOL` de F1, se elige el **más simple** (menos parámetros).
- Hay cola si `pct_posible_cola > 0.5`.

| Paso | Qué se prueba | Opciones | Base |
|---|---|---|---|
| E0 | GRU mínima | 1×GRU(32) → Dense(1, sigmoid), ventana 12, sin dropout | Mansur et al. (2026); Das (2023) |
| E1 | Unidades | 32 / 64 / 128 | Wang et al. (2021) |
| E2 | Largo de ventana | 12 / 24 / 48 / 96 bloques | Das (2023); Hewamalage et al. (2021) |
| E3 | Regularización | dropout 0 / 0.2 / 0.3 | sobreajuste visto en LSTM y GRU v1 |
| E4 | Profundidad | + Dense(32) / + 2.ª GRU / ambas | Das (2023); Greff et al. (2017) |
| E5 | Normalización y dirección | + LayerNorm / Bi-GRU | versiones `norm` y `bi` |
| Ref | Modelo v1 | 64→32 + Dense(32), ventana 48, dropout 0.2 | GRU anterior con TimeSeriesSplit |

Los resultados se van guardando en `experimentos_gru.csv`. Si Colab se desconecta, al volver a correr el notebook los experimentos ya hechos se saltan.

## 1. Configuración

In [ ]:
# ====== RUTAS (ajusta a tu Drive) ======
DATA_PATH = "/content/drive/MyDrive/Datos/Procesado/dataset_final_serie_tiempo.parquet"
MODELING_PARENT = "/content/modeling"
RESULTS_CSV = "/content/drive/MyDrive/Datos/Resultados/experimentos_gru.csv"   # tabla acumulada

# ====== REGLAS COMUNES ======
TARGET = "pct_posible_cola"
LABEL_THR = 0.5        # hay cola si pct_posible_cola > 0.5 (Diseño de Pruebas)
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
EPOCHS = 50
PATIENCE = 5           # early stopping sobre val_loss, restaurando los mejores pesos
BATCH = 128
LR = 1e-3
SEED = 42
TOL = 0.005            # si dos experimentos quedan a menos de TOL de F1 en val, gana el más simple
QUICK = False          # True = 3 épocas por experimento, solo para probar que todo corre
RERUN = False          # True = vuelve a entrenar aunque el experimento ya esté en el CSV
VERBOSE = 2            # 0 = silencioso | 2 = una línea por época

## 2. Imports y datos

In [ ]:
import sys, os, time, gc
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
# ---- Encontrar el paquete `modeling` (gru_model.py, tcn_model.py, model_base.py) ----
# 1) Si ya están en MODELING_PARENT/modeling/ se usan esos.
# 2) Si no, se clona el repo (rama REPO_BRANCH) y se usa vms/notebooks/modeling/.
#    OJO: split_temporal y v="custom" solo están en GitHub después de hacer commit + push.
REPO_URL = "https://github.com/kamimartinez/VMS-prediction.git"
REPO_BRANCH = "main"
NEEDED = ["gru_model.py", "tcn_model.py", "model_base.py"]

def _has_files(folder):
    return all(os.path.exists(os.path.join(folder, "modeling", f)) for f in NEEDED)

if not _has_files(MODELING_PARENT) and IN_COLAB:
    os.system(f"rm -rf /content/vms && git clone -q -b {REPO_BRANCH} {REPO_URL} /content/vms")
    if _has_files("/content/vms/notebooks"):
        MODELING_PARENT = "/content/vms/notebooks"
if not _has_files(MODELING_PARENT):
    raise FileNotFoundError(f"Faltan {NEEDED} en {MODELING_PARENT}/modeling/")
if MODELING_PARENT not in sys.path:
    sys.path.insert(0, MODELING_PARENT)
if not IN_COLAB:   # fuera de Colab, gru_model.py importa tcn_model directo
    sys.path.insert(0, os.path.join(MODELING_PARENT, "modeling"))
print("Usando módulos de:", os.path.join(MODELING_PARENT, "modeling"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, confusion_matrix)
from tensorflow.keras.callbacks import EarlyStopping

from modeling.gru_model import GRUModel

if not hasattr(GRUModel, "split_temporal"):
    raise ImportError("Esta versión de gru_model.py no tiene split_temporal; sube la versión nueva.")
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
df = pd.read_parquet(DATA_PATH)
df["temporada"] = df["temporada"].astype(object)   # prepare() espera texto tipo object
df["bloque_tiempo"] = pd.to_datetime(df["bloque_tiempo"])
print(df.shape)
df.head()

## 3. Split 70 / 15 / 15 y líneas base (paso 0)

Primero se revisa que el split por fila dé los mismos tamaños y fechas que *Diseño de Pruebas* (125,894 / 26,978 / 26,978) y que las referencias en test den lo mismo que en los demás modelos (**siempre cola F1 ≈ 0.730**, **bloque anterior F1 ≈ 0.667**). Si no coinciden, el test no es el mismo y hay que revisar antes de seguir.

In [ ]:
# Split a nivel fila (sin ventanas) para verificar contra Diseño de Pruebas
_ref = GRUModel()
rows = df[["ms", "bloque_tiempo"]].reset_index(drop=True)
r_tr, r_va, r_te = _ref.split_temporal(df, rows, TRAIN_FRAC, VAL_FRAC)
print("Filas:", _ref.split_info_["filas"])
for name, idx in [("Train", r_tr), ("Val", r_va), ("Test", r_te)]:
    t = rows["bloque_tiempo"].iloc[idx]
    print(f"{name:<5} | {t.min()} -> {t.max()} | {len(idx):,} bloques")

In [ ]:
def label(y):
    return (np.asarray(y, dtype=float) > LABEL_THR).astype(int)

def cola_metrics(y_true, y_pred_bin):
    yt = label(y_true); yp = np.asarray(y_pred_bin).astype(int)
    return {"F1": f1_score(yt, yp, zero_division=0),
            "Precision": precision_score(yt, yp, zero_division=0),
            "Recall": recall_score(yt, yp, zero_division=0),
            "pct_cola_real": yt.mean()}

# Persistencia: hay cola en t si la hubo en el bloque anterior (t-1) del mismo robot
d = df[["ms", "bloque_tiempo", TARGET]].copy()
d["__row__"] = np.arange(len(d))
d = d.sort_values(["ms", "bloque_tiempo"])
g = d.groupby("ms")
contiguo = (d["bloque_tiempo"] - g["bloque_tiempo"].shift(1)) == pd.Timedelta("30min")
d["prev"] = g[TARGET].shift(1).where(contiguo)
prev_by_row = d.set_index("__row__")["prev"]

# rows -> posición en df (rows viene de df.reset_index, así que el índice coincide)
def baselines(idx):
    y = rows.index[idx]
    y_true = df[TARGET].to_numpy()[y]
    prev = prev_by_row.loc[y].fillna(0).to_numpy()
    return pd.DataFrame({
        "Siempre cola": cola_metrics(y_true, np.ones(len(y))),
        "Bloque anterior": cola_metrics(y_true, label(prev)),
    }).T

BASE_VAL, BASE_TEST = baselines(r_va), baselines(r_te)
print("--- Validación ---"); display(BASE_VAL.round(3))
print("--- Test (solo para verificar que es el mismo test) ---"); display(BASE_TEST.round(3))

## 4. Funciones de los experimentos

- `get_data(window)`: arma las ventanas por robot (ventana en *t−1*, objetivo en *t*), hace el split y escala con train. Guarda en memoria solo la ventana actual (con 96 bloques las ventanas pesan ~2 GB).
- `run_experiment(...)`: entrena una configuración, elige el umbral con mejor F1 en validación y guarda una fila en el CSV.
- `pick_best(ids)`: gana el F1 de val más alto; empates (< `TOL`) se rompen por menos parámetros.

In [ ]:
_CACHE = {}

def get_data(window):
    if window in _CACHE:
        return _CACHE[window]
    _CACHE.clear(); gc.collect()
    prep = GRUModel(task="regression", window=window, threshold=LABEL_THR)
    X_seq, y_seq, meta = prep.prepare(df, df[TARGET])
    tr, va, te = prep.split_temporal(df, meta, TRAIN_FRAC, VAL_FRAC)
    X_tr, X_va, X_te = prep.scale(X_seq[tr], X_seq[va], X_seq[te])   # scaler ajustado SOLO con train
    del X_seq; gc.collect()
    _CACHE[window] = dict(prep=prep, X_tr=X_tr, X_va=X_va, X_te=X_te,
                          y_tr=y_seq[tr], y_va=y_seq[va], y_te=y_seq[te],
                          meta_va=meta.iloc[va].reset_index(drop=True),
                          meta_te=meta.iloc[te].reset_index(drop=True))
    print(f"Ventana {window}: train={len(tr):,} val={len(va):,} test={len(te):,} | features={X_tr.shape[-1]}")
    return _CACHE[window]

def best_threshold(y_true, y_prob, grid=np.arange(0.05, 0.96, 0.01)):
    yt = label(y_true)
    scores = [f1_score(yt, (y_prob >= t).astype(int), zero_division=0) for t in grid]
    i = int(np.argmax(scores))
    return round(float(grid[i]), 2)

CONFIG_KEYS = ["window", "units", "layers", "dropout", "dense", "norm", "bidirectional", "lr", "batch", "seed"]
RESULTS = pd.read_csv(RESULTS_CSV).to_dict("records") if os.path.exists(RESULTS_CSV) else []
RUNS = {}   # exp_id -> modelo, historia y predicciones de val (solo en esta sesión)

def _done(exp_id):
    return any(r["exp_id"] == exp_id for r in RESULTS)

def run_experiment(exp_id, stage, window=12, units=32, layers=1, dropout=0.0, dense=0,
                   norm=False, bidirectional=False, lr=LR, batch=BATCH, seed=SEED, note=""):
    if _done(exp_id) and not RERUN:
        print(f"{exp_id}: ya está en el CSV, se salta (RERUN=True para repetirlo)")
        return exp_id
    data = get_data(window)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    gru = GRUModel(task="regression", window=window, threshold=LABEL_THR, learning_rate=lr)
    model = gru.compile(input_shape=data["X_tr"].shape[1:], v="custom", units=units, layers=layers,
                        dropout=dropout, dense=dense, norm=norm, bidirectional=bidirectional)
    print(f"\n===== {exp_id} | {model.name} | ventana={window} | params={model.count_params():,} =====")
    t0 = time.time()
    hist = model.fit(data["X_tr"], data["y_tr"], validation_data=(data["X_va"], data["y_va"]),
                     epochs=3 if QUICK else EPOCHS, batch_size=batch, verbose=VERBOSE,
                     callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)])
    secs = time.time() - t0

    p_va = model.predict(data["X_va"], batch_size=1024, verbose=0).ravel()
    thr = best_threshold(data["y_va"], p_va)
    m = cola_metrics(data["y_va"], (p_va >= thr).astype(int))
    m05 = cola_metrics(data["y_va"], (p_va >= 0.5).astype(int))
    h = hist.history; best_ep = int(np.argmin(h["val_loss"]))

    row = {"exp_id": exp_id, "stage": stage, "modelo": model.name,
           "window": window, "units": units, "layers": layers, "dropout": dropout, "dense": dense,
           "norm": norm, "bidirectional": bidirectional, "lr": lr, "batch": batch, "seed": seed,
           "params": model.count_params(),
           "F1_val": m["F1"], "Precision_val": m["Precision"], "Recall_val": m["Recall"], "umbral": thr,
           "F1_val_05": m05["F1"],
           "MAE_val": mean_absolute_error(data["y_va"], p_va),
           "RMSE_val": float(np.sqrt(mean_squared_error(data["y_va"], p_va))),
           "loss_train": h["loss"][best_ep], "loss_val": h["val_loss"][best_ep],
           "brecha": h["val_loss"][best_ep] - h["loss"][best_ep],   # > 0 y creciendo = sobreajuste
           "mejor_epoca": best_ep + 1, "epocas": len(h["loss"]), "segundos": round(secs),
           "quick": QUICK, "nota": note}
    RESULTS[:] = [r for r in RESULTS if r["exp_id"] != exp_id] + [row]
    os.makedirs(os.path.dirname(RESULTS_CSV) or ".", exist_ok=True)
    pd.DataFrame(RESULTS).to_csv(RESULTS_CSV, index=False)
    RUNS[exp_id] = dict(model=model, history=h, p_va=p_va, thr=thr, window=window)
    print(f"{exp_id}: F1_val={m['F1']:.4f} (umbral {thr:.2f}) | P={m['Precision']:.3f} R={m['Recall']:.3f} | "
          f"MAE={row['MAE_val']:.4f} | mejor época {best_ep + 1}/{len(h['loss'])} | {secs:.0f}s")
    return exp_id

def config_of(exp_id):
    r = next(r for r in RESULTS if r["exp_id"] == exp_id)
    cfg = {k: r[k] for k in CONFIG_KEYS}
    for k in ["window", "units", "layers", "dense", "batch", "seed"]:
        cfg[k] = int(cfg[k])
    for k in ["norm", "bidirectional"]:
        cfg[k] = bool(cfg[k]) if not isinstance(cfg[k], str) else cfg[k] == "True"
    cfg["dropout"], cfg["lr"] = float(cfg["dropout"]), float(cfg["lr"])
    return cfg

def pick_best(ids):
    t = pd.DataFrame([r for r in RESULTS if r["exp_id"] in ids])
    top = t["F1_val"].max()
    best = t[t["F1_val"] >= top - TOL].sort_values(["params", "F1_val"], ascending=[True, False]).iloc[0]
    display(t[["exp_id", "modelo", "window", "params", "F1_val", "Precision_val", "Recall_val",
               "umbral", "MAE_val", "brecha", "mejor_epoca"]].sort_values("F1_val", ascending=False).round(4))
    print(f"Ganador: {best['exp_id']} (F1_val={best['F1_val']:.4f}, params={int(best['params']):,})")
    return best["exp_id"]

def plot_curves(ids):
    ids = [i for i in ids if i in RUNS]
    if not ids:
        print("No hay historias en esta sesión (los experimentos se cargaron del CSV)."); return
    fig, axes = plt.subplots(1, len(ids), figsize=(4.5 * len(ids), 3.5), squeeze=False)
    for ax, i in zip(axes[0], ids):
        h = RUNS[i]["history"]
        ax.plot(h["loss"], label="train"); ax.plot(h["val_loss"], label="val")
        ax.set_title(i); ax.set_xlabel("Época"); ax.legend()
    plt.tight_layout(); plt.show()

## 5. E0 — GRU mínima

1×GRU(32) → Dense(1, sigmoid), ventana de 12 bloques (6 h), sin dropout. Es el punto de partida; todo lo demás tiene que superarla en validación.

In [ ]:
best_id = run_experiment("E0", "E0", window=12, units=32, layers=1)
cfg = config_of(best_id)
print(BASE_VAL.round(3))   # referencias en val para comparar
plot_curves(["E0"])

## 6. E1 — Unidades (32 / 64 / 128)

In [ ]:
cands = [best_id]
for u in [64, 128]:
    cands.append(run_experiment(f"E1_u{u}", "E1", **{**cfg, "units": u}))
best_id = pick_best(cands); cfg = config_of(best_id)
plot_curves(cands)

## 7. E2 — Largo de ventana (12 / 24 / 48 / 96 bloques)

12 = 6 h, 24 = 12 h, 48 = 24 h (un día completo con los horarios de alimentación), 96 = 48 h. La ventana de 96 usa más RAM; si Colab se queda sin memoria, se puede quitar de la lista.

In [ ]:
cands = [best_id]
for w in [12, 24, 48, 96]:
    if w != cfg["window"]:
        cands.append(run_experiment(f"E2_w{w}", "E2", **{**cfg, "window": w}))
best_id = pick_best(cands); cfg = config_of(best_id)
plot_curves(cands)

## 8. E3 — Regularización (dropout 0 / 0.2 / 0.3)

Antes de correrlo, revisa las curvas del ganador de E2: si la pérdida de validación deja de bajar mientras la de train sigue bajando (columna `brecha` creciendo), hay sobreajuste y el dropout debería ayudar.

In [ ]:
cands = [best_id]
for p in [0.0, 0.2, 0.3]:
    if p != cfg["dropout"]:
        cands.append(run_experiment(f"E3_d{p:g}".replace(".", ""), "E3", **{**cfg, "dropout": p}))
best_id = pick_best(cands); cfg = config_of(best_id)
plot_curves(cands)

## 9. E4 — Profundidad (+ Dense(32) / + 2.ª capa GRU / ambas)

In [ ]:
cands = [best_id]
variantes = {"E4_dense": {"dense": 32},
             "E4_2gru": {"layers": 2},
             "E4_2gru_dense": {"layers": 2, "dense": 32}}
for exp_id, cambio in variantes.items():
    cands.append(run_experiment(exp_id, "E4", **{**cfg, **cambio}))
best_id = pick_best(cands); cfg = config_of(best_id)
plot_curves(cands)

## 10. E5 — Normalización y dirección (+ LayerNorm / Bi-GRU)

In [ ]:
cands = [best_id]
variantes = {"E5_norm": {"norm": True},
             "E5_bi": {"bidirectional": True}}
for exp_id, cambio in variantes.items():
    cands.append(run_experiment(exp_id, "E5", **{**cfg, **cambio}))
best_id = pick_best(cands); cfg = config_of(best_id)
plot_curves(cands)

## 11. Ref — Modelo v1 con el split nuevo

La GRU anterior (64→32 + Dense(32), ventana 48, dropout 0.2), ahora con 70/15/15. Sirve para ver dónde queda contra la escalera; no entra en la selección.

In [ ]:
run_experiment("REF_v1", "REF", window=48, units=64, layers=2, dropout=0.2, dense=32,
               note="GRU v1 (antes con TimeSeriesSplit)")

## 12. Resumen de la primera tanda

In [ ]:
res = pd.DataFrame(RESULTS)
cols = ["exp_id", "stage", "modelo", "window", "params", "F1_val", "Precision_val", "Recall_val",
        "umbral", "F1_val_05", "MAE_val", "RMSE_val", "brecha", "mejor_epoca", "epocas", "segundos"]
display(res[cols].round(4))
print("Modelo elegido hasta ahora:", best_id, cfg)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ["tab:green" if i == best_id else ("tab:gray" if s == "REF" else "tab:blue")
          for i, s in zip(res["exp_id"], res["stage"])]
ax.bar(res["exp_id"], res["F1_val"], color=colors)
ax.axhline(BASE_VAL.loc["Siempre cola", "F1"], color="orange", ls="--", label="Siempre cola (val)")
ax.axhline(BASE_VAL.loc["Bloque anterior", "F1"], color="purple", ls="--", label="Bloque anterior (val)")
ax.axhline(0.60, color="red", ls=":", label="Criterio F1 = 0.60")
ax.set_ylabel("F1 en validación"); ax.set_title("Escalera de experimentos GRU")
plt.xticks(rotation=45, ha="right"); ax.legend(); plt.tight_layout(); plt.show()

## 13. Evaluación en test

Evalúa en test el ganador de la primera tanda (E0–E5) contra *siempre cola* y *bloque anterior*. Los resultados de test **no se usan para elegir nada**: la selección de E0–E5 (y de E6–E9 si se hacen) se queda con validación.

In [ ]:
EVALUAR_TEST = True

if EVALUAR_TEST:
    if best_id not in RUNS:   # si el ganador se cargó del CSV, se vuelve a entrenar con la misma semilla
        RERUN = True; run_experiment(best_id, "FINAL", **cfg); RERUN = False
    data = get_data(cfg["window"]); run = RUNS[best_id]
    p_te = run["model"].predict(data["X_te"], batch_size=1024, verbose=0).ravel()
    m_te = cola_metrics(data["y_te"], (p_te >= run["thr"]).astype(int))
    tabla_test = pd.concat([BASE_TEST, pd.DataFrame({f"GRU ({best_id})": m_te}).T])
    display(tabla_test.round(3))
    print("Umbral (elegido en val):", run["thr"])
    print("MAE test:", round(mean_absolute_error(data["y_te"], p_te), 4),
          "| RMSE test:", round(float(np.sqrt(mean_squared_error(data["y_te"], p_te))), 4))
    print("Matriz de confusión (filas = real, columnas = predicho):")
    print(confusion_matrix(label(data["y_te"]), (p_te >= run["thr"]).astype(int)))

## 14. Predicción vs. real en una semana (un robot, test)

In [ ]:
if EVALUAR_TEST:
    m = data["meta_te"].copy()
    m["real"] = data["y_te"]; m["pred"] = p_te
    robot = m["ms"].iloc[0]
    s = m[m["ms"] == robot].head(336)          # 336 bloques = 7 días
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(s["bloque_tiempo"], s["real"], label="Real", alpha=0.7)
    ax.plot(s["bloque_tiempo"], s["pred"], label="Predicho", alpha=0.9)
    ax.axhline(run["thr"], color="red", ls=":", label=f"Umbral {run['thr']:.2f}")
    ax.set_title(f"{robot}: primera semana del test"); ax.legend()
    plt.tight_layout(); plt.show()

## 15. Importancia de variables (por permutación, test)

Desordena una variable a la vez (en todos los pasos de la ventana) y mide cuánto sube el MAE y cuánto baja el F1. Más cambio = más importante. Útil para ver si las variables de lactancia (`pct_fresca`, `pct_alta`, `pct_baja`, `pct_extendida`, `pct_primer_parto`) aportan algo. Se muestrea para que corra rápido.

In [ ]:
if EVALUAR_TEST:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(data["X_te"]), size=min(5000, len(data["X_te"])), replace=False)
    Xs, ys = data["X_te"][idx], data["y_te"][idx]
    def _score(X):
        p = run["model"].predict(X, batch_size=1024, verbose=0).ravel()
        return mean_absolute_error(ys, p), cola_metrics(ys, (p >= run["thr"]).astype(int))["F1"]
    mae0, f10 = _score(Xs)
    imp = {}
    for j, name in enumerate(data["prep"].feature_names_):
        Xp = Xs.copy()
        Xp[:, :, j] = Xp[rng.permutation(len(Xs)), :, j]
        mae, f1 = _score(Xp)
        imp[name] = {"aumento_MAE": mae - mae0, "caida_F1": f10 - f1}
    imp = pd.DataFrame(imp).T.sort_values("aumento_MAE")
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    imp["aumento_MAE"].plot.barh(ax=axes[0], title="Aumento en MAE al permutar")
    imp["caida_F1"].sort_values().plot.barh(ax=axes[1], title="Caída en F1 al permutar")
    plt.tight_layout(); plt.show()

## 16. Exportar el modelo

In [ ]:
if EVALUAR_TEST:
    import json, joblib
    from pathlib import Path
    MODELS_DIR = Path(MODELING_PARENT) / "models"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    tag = f"gru_{best_id}" + ("_quick" if QUICK else "")
    model_file, scaler_file, config_file = (MODELS_DIR / f"{tag}.keras",
                                            MODELS_DIR / f"{tag}_scaler.joblib",
                                            MODELS_DIR / f"{tag}_config.json")
    run["model"].save(model_file)
    joblib.dump(data["prep"].scaler, scaler_file)
    fila = next(r for r in RESULTS if r["exp_id"] == best_id)
    json.dump({
        "exp_id": best_id, "task": "regression", "target": TARGET, **cfg,
        "threshold_label": LABEL_THR, "threshold_decision": run["thr"],
        "feature_names": data["prep"].feature_names_,
        "input_shape": list(data["X_te"].shape[1:]), "shift_target": True,
        "split": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": round(1 - TRAIN_FRAC - VAL_FRAC, 2)},
        "metrics_val": {k: fila[k] for k in ["F1_val", "Precision_val", "Recall_val", "MAE_val", "RMSE_val"]},
        "metrics_test": {**m_te, "MAE": float(mean_absolute_error(data["y_te"], p_te))},
    }, open(config_file, "w"), indent=2, ensure_ascii=False, default=float)
    print("Guardado:")
    for p in (model_file, scaler_file, config_file):
        print(" ", p)

## 17. Verificar exportación

In [ ]:
if EVALUAR_TEST:
    loaded = tf.keras.models.load_model(model_file)
    p_orig = run["model"].predict(data["X_te"][:1000], verbose=0).ravel()
    p_load = loaded.predict(data["X_te"][:1000], verbose=0).ravel()
    print("Máx. diferencia:", float(np.abs(p_orig - p_load).max()))   # debe ser ~0